In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from share.sumcheck import Verifier, Prover
from share.mlesumcheck import MLEProver, trans_tables
from share.interactive import Simulation
from share.field import F
from share.poly import UniPoly
from share.mle import MLE

from itertools import product
from random import randint, seed
from types import MethodType

In [2]:
tables = [[1, 2, 3, 4, 1, 2, 3, 4], [5, 6, 7, 8, 5, 6, 7, 8]]
g, degs = trans_tables(tables)

In [3]:
prover = Prover("Prover", g, degs)
verifier = Verifier("Verifier", g, degs, prover.sum())
prover.set_dst(verifier)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()

{'Prover': None, 'Verifier': True}
Prover -> Verifier: ('g1', [F(70), F(70), F(70)])
Verifier -> Prover: ('r1', F(2134139354768280835))
Prover -> Verifier: ('g2', [F(17), F(53), F(105)])
Verifier -> Prover: ('r2', F(1636871794675947429))
Prover -> Verifier: ('g3', [F(484367316484624207), F(114325467547332077), F(2050126627823733900)])
Prover         1.965 ms  (3 activations)
Verifier       1.258 ms  (4 activations)


In [4]:
mleprover = MLEProver("MLEProver", tables)
verifier.claims = [mleprover.sum()]
mleprover.set_dst(verifier)

seed(1337)
sim = Simulation([mleprover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()

{'MLEProver': None, 'Verifier': True}
MLEProver -> Verifier: ('g1', [70, 70, 70])
Verifier -> MLEProver: ('r1', F(2134139354768280835))
MLEProver -> Verifier: ('g2', [F(17), F(53), F(105)])
Verifier -> MLEProver: ('r2', F(1636871794675947429))
MLEProver -> Verifier: ('g3', [F(484367316484624207), F(114325467547332077), F(2050126627823733900)])
MLEProver       0.412 ms  (3 activations)
Verifier        0.629 ms  (4 activations)


In [43]:
from share.matmul import MatmulProver, MatmulVerifier

A = list(range(256))
B = list(range(256, 512))

mp = MatmulProver("MP", A, B)
mv = MatmulVerifier("MV", ([F(19937)]*4, [F(65537)]*4), A, B)
mv.set_dst(mp)

seed(1337)
sim = Simulation([mp, mv])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()

{'MP': None, 'MV': True}
MV -> MP: ([F(19937), F(19937), F(19937), F(19937)], [F(65537), F(65537), F(65537), F(65537)])
MP -> MV: F(75289707189640)
MP -> MV: ('g1', [F(37642372262628), F(37647334927012), F(37652297607780)])
MV -> MP: ('r1', F(2134139354768280835))
MP -> MV: ('g2', [F(411011924588976847), F(394961127594172814), F(378910330599370829)])
MV -> MP: ('r2', F(1636871794675947429))
MP -> MV: ('g3', [F(1329310662428002716), F(1806183740390057399), F(2283056818352112338)])
MV -> MP: ('r3', F(1316587892191542527))
MP -> MV: ('g4', [F(1300782567417792830), F(2094816853170622473), F(583008129709758197)])
MP       7.152 ms  (5 activations)
MV       4.556 ms  (6 activations)


In [51]:
mle_A = MLE(A)
mle_B = MLE(B)
g = lambda *z: mle_A(*[F(19937)]*4, *z) * mle_B(*z, *[F(65537)]*4)
degs = [2, 2, 2, 2]

prover = Prover("P", g, degs)
verifier = Verifier("V", g, degs, prover.sum())
prover.set_dst(verifier)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()

{'P': None, 'V': True}
P -> V: ('g1', [F(37642372262628), F(37647334927012), F(37652297607780)])
V -> P: ('r1', F(2134139354768280835))
P -> V: ('g2', [F(411011924588976847), F(394961127594172814), F(378910330599370829)])
V -> P: ('r2', F(1636871794675947429))
P -> V: ('g3', [F(1329310662428002716), F(1806183740390057399), F(2283056818352112338)])
V -> P: ('r3', F(1316587892191542527))
P -> V: ('g4', [F(1300782567417792830), F(2094816853170622473), F(583008129709758197)])
P      53.171 ms  (4 activations)
V       1.197 ms  (5 activations)
